# Pizza Agent: construye un agente de IA paso a paso


**Objetivo:** completar un agente que interprete un objetivo, elija una herramienta, la ejecute, observe el resultado, decida el siguiente paso y termine cuando resuelva la tarea.

> **Un LLM es una función. Un agente es un bucle.**


## El reto

> **"Quiero una pizza Jurassic Park grande. Comprueba si está disponible, calcula el precio y dime si puedo pedirla."**

<table style="border: none; width: 100%;">
  <tr style="border: none;">
    <td style="border: none; text-align: center; width: 33%;">
      <img src="https://asiersanz.com/wp-content/uploads/2026/05/20241127_124425.jpg?w=765" style="width: 200px; height: 120px; object-fit: cover; border-radius: 4px;" />
    </td>
    <td style="border: none; text-align: center; width: 33%;">
      <div style="display: inline-block; width: 200px; height: 120px; overflow: hidden; border-radius: 4px;">
        <img src="https://asiersanz.com/wp-content/uploads/2026/05/20241127_124343.jpg?w=765" style="width: 200px; height: auto; min-height: 120px; object-fit: cover; object-position: center;" />
      </div>
    </td>
    <td style="border: none; text-align: center; width: 33%;">
      <img src="https://asiersanz.com/wp-content/uploads/2026/05/20241127_161229-1.jpg?w=765" style="width: 200px; height: 120px; object-fit: cover; border-radius: 4px;" />
    </td>
  </tr>
</table>

Ruta esperada:

```text
Usuario → LLM → comprobar_disponibilidad → OBSERVACIÓN
                    ↓
                  LLM → calcular_precio → OBSERVACIÓN
                    ↓
                  LLM → respuesta final
```

No se va a utilizar LangChain ni LangGraph. Vas a construir a mano el motor principal del agente.

# 1. Preparación

Este laboratorio funciona en **dos modos**:

- **Modo real:** si tienes `GROQ_API_KEY`, el agente utiliza un LLM real con tool calling.
- **Modo demo:** si no tienes API key, una simulación programada representa algunas respuestas del modelo. Permite practicar el flujo, pero **las decisiones no las toma un LLM real**.

### Si quieres usar el modo real:

1. Crea una cuenta gratuita en [GroqCloud](https://console.groq.com/).
2. Abre **API Keys**: https://console.groq.com/keys
3. Pulsa **Create API Key**.
4. En Google Colab abre el panel **Secrets** (icono de llave).
5. Crea un secreto llamado `GROQ_API_KEY`.
6. Activa **Notebook access**.
7. No escribas la clave directamente en el código.

Si no tienes API key, **no pasa nada**: deja el secreto sin crear y el notebook utilizará automáticamente el modo demo.


In [ ]:
%pip -q install -U groq


In [ ]:
import os
from types import SimpleNamespace

try:
    from google.colab import userdata
    API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    API_KEY = os.environ.get("GROQ_API_KEY")

DEMO_MODE = not bool(API_KEY)

if DEMO_MODE:
    client = None
    print("⚠️ MODO DEMO")
    print("No se ha encontrado GROQ_API_KEY.")
    print("El agente utilizará una simulación inteligente para practicar el flujo de tool calling.")
else:
    from groq import Groq
    client = Groq(api_key=API_KEY)
    print("✅ MODO REAL: Groq API")

MODEL = "openai/gpt-oss-120b"


# 2. El mundo del agente

Estas herramientas ya están programadas. Tu trabajo será conectarlas al LLM.

- `comprobar_disponibilidad`
- `calcular_precio`
- `realizar_pedido`

La última representa una acción externa sensible. **En este notebook no se implementa una aprobación humana real**: en una aplicación real, habría que añadir una confirmación explícita antes de ejecutar un pedido.


In [ ]:
MENU = {
    "jurassic park": {"precios": {"mediana": 12.0, "grande": 15.0}},
    "et": {"precios": {"mediana": 10.0, "grande": 13.0}},
    "indiana jones": {"precios": {"mediana": 11.0, "grande": 14.0}},
}

AVAILABLE = {
    ("jurassic park", "mediana"),
    ("jurassic park", "grande"),
    ("et", "mediana"),
    ("et", "grande"),
    ("indiana jones", "mediana"),
}

ORDERS = []

def comprobar_disponibilidad(pizza: str, tamaño: str):
    return {
        "pizza": pizza,
        "tamaño": tamaño,
        "disponible": (pizza.lower(), tamaño.lower()) in AVAILABLE,
    }

def calcular_precio(items: list):
    total = 0.0
    for item in items:
        pizza = item["pizza"].lower()
        tamaño = item["tamaño"].lower()
        cantidad = int(item.get("cantidad", 1))
        total += MENU[pizza]["precios"].get(tamaño, 0) * cantidad
    return {"items": items, "total": round(total, 2)}

def realizar_pedido(items: list, total: float):
    order = {
        "order_id": f"DEMO-{len(ORDERS)+1:03d}",
        "items": items,
        "total": total,
        "status": "confirmed",
    }
    ORDERS.append(order)
    return order

# 3. Tools

El modelo necesita conocer las herramientas y sus parámetros. Esto se describe con un **tool schema**.

Observa especialmente las *descripciones*: son parte del contrato entre el modelo y el código.

En este notebook utilizaremos el formato de herramientas de la API de Groq, compatible con el esquema habitual de tool calling.

**Cómo funciona:** el modelo no ejecuta directamente las funciones Python. Devuelve un `tool_call` que indica el nombre de la herramienta y sus argumentos. La aplicación lee esa petición, ejecuta la función correspondiente y devuelve el resultado al modelo. Con esa observación, el modelo puede decidir qué hacer después.


In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "comprobar_disponibilidad",
            "description": "Comprueba si una pizza está disponible en un tamaño concreto.",
            "parameters": {
                "type": "object",
                "properties": {
                    "pizza": {"type": "string", "description": "Nombre de la pizza."},
                    "tamaño": {"type": "string", "description": "Tamaño: mediana o grande."},
                },
                "required": ["pizza", "tamaño"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calcular_precio",
            "description": "Calcula el precio total de una lista de pizzas.",
            "parameters": {
                "type": "object",
                "properties": {
                    "items": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "pizza": {"type": "string"},
                                "tamaño": {"type": "string"},
                                "cantidad": {"type": "integer"},
                            },
                            "required": ["pizza", "tamaño", "cantidad"],
                        },
                    }
                },
                "required": ["items"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "realizar_pedido",
            "description": (
                "Simula la confirmación de un pedido. Es una acción externa sensible "
                "y requiere aprobación humana."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "items": {"type": "array"},
                    "total": {"type": "number"},
                },
                "required": ["items", "total"],
            },
        },
    },
]

print("Tools declaradas:", [t["function"]["name"] for t in TOOLS])


## Funciones auxiliares del modo demo

Estas funciones **simulan las respuestas del modelo cuando no hay una API key**. Sirven para que puedas practicar el flujo de llamadas a herramientas sin conectarte a un modelo real.

**Para entender cómo funciona el agente, no hace falta detenerse en estas funciones.** Puedes seguir con los ejercicios y volver aquí solo si quieres ver en detalle cómo se simula cada respuesta en modo demo.


In [ ]:
import json

# Respuestas simuladas para el Modo Demo
def _mock_response_disponibilidad(pizza="Jurassic Park", tamaño="grande"):
    """Simula que el modelo decide comprobar el tamaño solicitado."""
    return SimpleNamespace(
        choices=[
            SimpleNamespace(
                message=SimpleNamespace(
                    role="assistant",
                    content=None,
                    tool_calls=[
                        SimpleNamespace(
                            id="demo_call_1",
                            type="function",
                            function=SimpleNamespace(
                                name="comprobar_disponibilidad",
                                arguments=json.dumps(
                                    {"pizza": pizza, "tamaño": tamaño},
                                    ensure_ascii=False,
                                ),
                            ),
                        )
                    ],
                )
            )
        ]
    )

def _mock_response_precio(pizza="Jurassic Park", tamaño="grande"):
    """Simula el cálculo del precio para el tamaño solicitado."""
    return SimpleNamespace(
        choices=[
            SimpleNamespace(
                message=SimpleNamespace(
                    role="assistant",
                    content=None,
                    tool_calls=[
                        SimpleNamespace(
                            id="demo_call_2",
                            type="function",
                            function=SimpleNamespace(
                                name="calcular_precio",
                                arguments=json.dumps(
                                    {"items": [{"pizza": pizza, "tamaño": tamaño, "cantidad": 1}]},
                                    ensure_ascii=False,
                                ),
                            ),
                        )
                    ],
                )
            )
        ]
    )

def _mock_response_final(pizza="Jurassic Park", tamaño="grande"):
    """Simula la respuesta final después de comprobar disponibilidad y precio."""
    precios = {
        ("jurassic park", "mediana"): 12.0,
        ("jurassic park", "grande"): 15.0,
        ("et", "mediana"): 10.0,
        ("et", "grande"): 13.0,
        ("indiana jones", "mediana"): 11.0,
    }
    precio = precios.get((pizza.lower(), tamaño.lower()))
    if precio is None:
        respuesta = f"No tengo un precio fiable para la pizza {pizza} {tamaño}."
    else:
        respuesta = (
            f"¡Sí! La pizza {pizza} {tamaño} está disponible "
            f"y cuesta {precio:.2f} €. ¿Confirmo el pedido?"
        )
    return SimpleNamespace(
        choices=[
            SimpleNamespace(
                message=SimpleNamespace(
                    role="assistant",
                    content=respuesta,
                    tool_calls=None,
                )
            )
        ]
    )

def _mock_response_aclaracion():
    """No inventa un precio si el tamaño pedido no está disponible."""
    return SimpleNamespace(
        choices=[
            SimpleNamespace(
                message=SimpleNamespace(
                    role="assistant",
                    content=(
                        "La pizza Jurassic Park pequeña no está disponible "
                        "en el menú y no tengo un precio para ese tamaño. "
                        "Solo puedo ofrecerte la mediana o la grande. ¿Cuál prefieres?"
                    ),
                    tool_calls=None,
                )
            )
        ]
    )


def mock_call_model(messages):
    """Simula la siguiente respuesta del modelo en modo demo."""
    tool_results = [m for m in messages if m.get("role") == "tool"]
    user_goal = next(
        (m.get("content", "") for m in reversed(messages) if m.get("role") == "user"),
        "",
    ).lower()

    # Primera iteración: comprobar el tamaño que pidió el usuario.
    if not tool_results:
        tamaño = "pequeña" if "pequeñ" in user_goal else (
            "mediana" if "mediana" in user_goal else "grande"
        )
        return _mock_response_disponibilidad("Jurassic Park", tamaño)

    # Si el tamaño no está disponible, no se calcula ni se inventa un precio.
    ultima_observacion = json.loads(tool_results[-1].get("content", "{}"))
    if ultima_observacion.get("disponible") is False:
        return _mock_response_aclaracion()

    # Segunda iteración: calcular el precio.
    if len(tool_results) == 1:
        tamaño = "mediana" if "mediana" in user_goal else "grande"
        return _mock_response_precio("Jurassic Park", tamaño)

    # Después del resultado del precio, dar la respuesta final.
    tamaño = "mediana" if "mediana" in user_goal else "grande"
    return _mock_response_final("Jurassic Park", tamaño)


# 4. Ejercicio 1 — El modelo elige una herramienta

Haz que el agente reciba:

> **"¿Tienen pizzas Jurassic Park grandes?"**

### TODO 1

Completa la llamada para que pueda utilizar `TOOLS`.

Después localiza el bloque `tool_calls`.


In [ ]:
SYSTEM_PROMPT = (
    "Eres el asistente de una pizzería. "
    "Cuando el usuario pregunte por la disponibilidad de una pizza y un tamaño concreto, "
    "debes utilizar siempre comprobar_disponibilidad antes de responder. "
    "No inventes precios ni disponibilidad. "
    "Si la herramienta indica que el tamaño no está disponible, "
    "no utilices calcular_precio y responde indicando los tamaños disponibles."
)

def call_model(messages, tools=TOOLS):
    """Usa el modelo real o delega en la simulación."""
    if DEMO_MODE:
        return mock_call_model(messages)

    return client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools,
        tool_choice="auto",
        temperature=0.2,
    )

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "¿Tienen pizzas Jurassic Park grandes?"},
]

# TODO 1: llama al modelo usando el historial messages y las herramientas TOOLS
response = None

# Groq devuelve las llamadas a función dentro de message.tool_calls
message = response.choices[0].message

if message.tool_calls:
    for tool_call in message.tool_calls:
        print("tool_call:", tool_call.function.name)
        print("arguments:", tool_call.function.arguments)
else:
    print("text:", message.content)


Esperamos un `tool_call` con:

```text
name = comprobar_disponibilidad
args = {"pizza": "Jurassic Park", "tamaño": "grande"}
```

**El modelo ha decidido qué hacer. La aplicación todavía tiene que ejecutar la herramienta.** El `tool_call` es una petición estructurada, no el resultado de la acción.


# 5. Ejercicio 2 — Ejecutar la herramienta

Ahora conecta la decisión del LLM con Python.

### TODO 2

Completa `execute_tool()`:

1. comprueba que la herramienta existe;
2. ejecuta la función;
3. devuelve el resultado.


In [ ]:
TOOL_FUNCTIONS = {
    "comprobar_disponibilidad": comprobar_disponibilidad,
    "calcular_precio": calcular_precio,
    "realizar_pedido": realizar_pedido,
}

def execute_tool(name, arguments):
    if name not in TOOL_FUNCTIONS:
        raise ValueError(f"Herramienta desconocida: {name}")
    # TODO 2: ejecuta la función correspondiente con arguments y devuelve su resultado
    pass

print(execute_tool(
    "comprobar_disponibilidad",
    {"pizza": "Jurassic Park", "tamaño": "grande"}
))


El resultado esperado es una observación similar a:

```text
{'pizza': 'Jurassic Park', 'tamaño': 'grande', 'disponible': True}
```

Esta respuesta de la herramienta se llama **observación**. No es todavía la respuesta final al usuario: se añade al historial y se envía al modelo para que pueda decidir el siguiente paso basándose en un resultado real del código, en lugar de inventárselo.


# 6. Ejercicio 3 — Construir el Agent Loop

Completa los `TODO` de `run_agent()`.

El ciclo es:

```text
LLM → tool_call → ejecutar → tool_result → LLM → ...
```

El estado se conserva en `messages`: es el historial que reúne la petición del usuario, las decisiones del asistente y los resultados de las herramientas. Al añadir cada mensaje, el modelo recibe el contexto de lo que ya ocurrió en las iteraciones anteriores.

**No programes qué herramienta usar para cada pregunta. El modelo debe decidirlo.**


In [ ]:
import json

def run_agent(goal, max_iterations=5):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": goal},
    ]

    for iteration in range(max_iterations):

        # TODO 3A: llama al modelo con tools=TOOLS
        response = None  # Sustituye esta línea al completar el TODO 3A
        message = response.choices[0].message

        # TODO 3B: obtiene las llamadas a las funciones
        tool_calls = []  # Sustituye esta línea al completar el TODO 3B

        if not tool_calls:
            print("\nAGENTE → RESPUESTA FINAL")
            print(message.content)
            return

        # Se conserva la respuesta del modelo en el historial
        assistant_message = {
            "role": "assistant",
            "content": message.content,
            "tool_calls": [
                {
                    "id": tc.id,
                    "type": "function",
                    "function": {
                        "name": tc.function.name,
                        "arguments": tc.function.arguments,
                    },
                }
                for tc in tool_calls
            ],
        }
        messages.append(assistant_message)

        for tool_call in tool_calls:
            print(f"\nITERACIÓN {iteration + 1}")
            print("ACTION:", tool_call.function.name)

            arguments = json.loads(tool_call.function.arguments)
            print("INPUT:", arguments)

            observation = execute_tool(
                tool_call.function.name,
                arguments
            )

            print("OBSERVATION:", observation)

            # TODO 3C: añade la observación al historial como mensaje de tipo tool



    print("\nSTOP: se alcanzó max_iterations.")


<details>
<summary>Ayuda para los TODO 3A–3C</summary>

### 3A — llamar al modelo

```python
response = call_model(messages, tools=TOOLS)
```

### 3B — detectar las herramientas solicitadas

```python
tool_calls = message.tool_calls or []
```

### 3C — devolver una observación

```python
messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": json.dumps(observation, ensure_ascii=False),
})
```

</details>


# 7. Ejecuta tu agente

Si has completado los TODO, ejecuta:

> **"Quiero una pizza Jurassic Park grande. Comprueba si está disponible, calcula el precio y dime si puedo pedirla."**


In [ ]:
run_agent(
    "Quiero una pizza Jurassic Park grande. "
    "Comprueba si está disponible, calcula el precio y dime si puedo pedirla."
)


### Observa la traza

El recorrido debería ser conceptualmente:

```text
ACTION: comprobar_disponibilidad
OBSERVATION: disponible=True

ACTION: calcular_precio
OBSERVATION: total=15.0

AGENTE → RESPUESTA FINAL
```

**La segunda decisión depende de la primera observación.**

Esa es la diferencia clave entre una consulta simple y un agente.


<div style="text-align: center; margin-top: 20px;">
  <img src="https://www.ecartelera.com/images/img/255400/255432_pizza-movies.webp" style="max-width: 100%; height: auto; border-radius: 8px;" />
</div>

# 8. Bonus

Cambia el objetivo por:

> **"Quiero una pizza Jurassic Park pequeña. Comprueba si está disponible y dime el precio."**

### Pregunta

¿Qué debería hacer un agente fiable?

- ¿Inventarse un precio?
- ¿Utilizar una herramienta?
- ¿Pedir aclaración?
- ¿Detenerse?


In [ ]:
run_agent(
    "Quiero una pizza Jurassic Park pequeña. "
    "Comprueba si está disponible y dime el precio."
)


# 9. Conclusión

Has construido:

| Concepto | Implementación |
|---|---|
| Objetivo | petición del usuario |
| Modelo | Groq |
| Tools | funciones Python |
| Decisión | `tool_call` |
| Acción | `execute_tool()` |
| Observación | mensaje `tool` |
| Estado | `messages` |
| Bucle | `run_agent()` |
| Parada | respuesta / `max_iterations` |

<br>

## La idea que debes recordar

> **Un agente utiliza un modelo para decidir el siguiente paso, ejecuta ese paso, observa el resultado y vuelve a decidir.**
